In [1]:
from __future__ import annotations

from pathlib import Path

import mne
import numpy as np
import pandas as pd


PROJECT_ROOT: Path = Path("/home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP")

BDF_PATH: Path = PROJECT_ROOT / "dataset" / "raw" / "bdf" / "s28.bdf"
RATINGS_PATH: Path = PROJECT_ROOT / "dataset" / "raw" / "metadata" / "participant_ratings.xls"

ORIGINAL_SFREQ: float = 512.0

In [2]:
raw: mne.io.BaseRaw = mne.io.read_raw_bdf(
    BDF_PATH,
    preload=False,
    verbose=False,
)

print("BDF:", BDF_PATH)
print("sfreq:", raw.info["sfreq"])
print("n_times:", raw.n_times)
print("duration_sec:", raw.n_times / raw.info["sfreq"])
print("last channels:", raw.ch_names[-10:])

BDF: /home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP/dataset/raw/bdf/s28.bdf
sfreq: 512.0
n_times: 1805824
duration_sec: 3527.0
last channels: ['EXG7', 'EXG8', 'GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp', '']


/tmp/ipykernel_21774/3264834722.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw: mne.io.BaseRaw = mne.io.read_raw_bdf(
/tmp/ipykernel_21774/3264834722.py:1: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw: mne.io.BaseRaw = mne.io.read_raw_bdf(


In [3]:
def normalize_event_codes(event_codes: np.ndarray) -> np.ndarray:
    normalized_codes: np.ndarray = event_codes.copy().astype(int)
    mask: np.ndarray = normalized_codes >= 1638144
    normalized_codes[mask] = normalized_codes[mask] - 1638144
    return normalized_codes


for channel_name in raw.ch_names[-10:]:
    try:
        events: np.ndarray = mne.find_events(
            raw,
            stim_channel=channel_name,
            shortest_event=1,
            verbose=False,
        )

        if events.size == 0:
            continue

        normalized_codes: np.ndarray = normalize_event_codes(events[:, 2])
        unique_codes: np.ndarray
        counts: np.ndarray
        unique_codes, counts = np.unique(normalized_codes, return_counts=True)

        print("\nCHANNEL:", repr(channel_name))
        print("num_events:", len(events))
        print(dict(zip(unique_codes.tolist(), counts.tolist())))

    except Exception as error:
        print("ERROR:", repr(channel_name), error)


CHANNEL: 'GSR1'
num_events: 864849
{91000: 2, 91001: 2, 91002: 3, 91003: 3, 91004: 3, 91005: 5, 91006: 8, 91007: 7, 91008: 4, 91009: 5, 91010: 11, 91011: 11, 91012: 4, 91013: 7, 91014: 3, 91015: 11, 91016: 10, 91017: 6, 91018: 1, 91019: 4, 91020: 9, 91021: 2, 91022: 4, 91023: 6, 91024: 2, 91025: 1, 91026: 4, 91027: 4, 91028: 3, 91029: 5, 91030: 2, 91031: 3, 91032: 2, 91033: 3, 91034: 2, 91035: 1, 91036: 1, 91037: 2, 91038: 4, 91039: 4, 91040: 1, 91041: 2, 91042: 2, 91043: 4, 91044: 1, 91045: 5, 91046: 1, 91047: 1, 91048: 3, 91049: 3, 91050: 6, 91051: 3, 91053: 3, 91054: 3, 91055: 2, 91056: 3, 91057: 1, 91058: 1, 91060: 3, 91061: 1, 91062: 1, 91063: 4, 91064: 2, 91065: 2, 91067: 3, 91069: 1, 91070: 4, 91071: 2, 91073: 2, 91074: 1, 91075: 2, 91076: 2, 91077: 2, 91078: 6, 91079: 1, 91080: 2, 91082: 3, 91083: 1, 91084: 2, 91085: 1, 91086: 1, 91087: 2, 91088: 1, 91089: 2, 91091: 1, 91092: 2, 91093: 2, 91094: 1, 91095: 1, 91097: 1, 91098: 1, 91099: 2, 91101: 3, 91102: 1, 91104: 2, 91106: 1,

In [4]:
STATUS_CHANNEL: str = ""  # cambia si el canal correcto tiene otro nombre

events: np.ndarray = mne.find_events(
    raw,
    stim_channel=STATUS_CHANNEL,
    shortest_event=1,
    verbose=False,
)

events[:, 2] = normalize_event_codes(events[:, 2])

relevant_events: np.ndarray = events[np.isin(events[:, 2], [3, 4, 5])]

df_events: pd.DataFrame = pd.DataFrame(
    {
        "sample": relevant_events[:, 0].astype(int),
        "time_sec": relevant_events[:, 0] / ORIGINAL_SFREQ,
        "code": relevant_events[:, 2].astype(int),
    }
)

print("counts:")
print(df_events["code"].value_counts().sort_index())

display(df_events.head(40))
display(df_events.tail(40))

counts:
code
3    39
4    37
5    39
Name: count, dtype: int64


,sample,time_sec,code
0,14391,28.107422,3
1,25626,50.050781,5
2,41049,80.173828,5
3,109786,214.425781,3
4,112371,219.474609,4
5,143140,279.570312,5
6,153863,300.513672,3
7,156439,305.544922,4
8,187191,365.607422,5
9,196497,383.783203,3


,sample,time_sec,code
75,1260519,2461.951172,5
76,1266422,2473.480469,3
77,1269007,2478.529297,4
78,1299750,2538.574219,5
79,1309193,2557.017578,3
80,1311778,2562.066406,4
81,1342522,2622.113281,5
82,1350037,2636.791016,3
83,1352622,2641.839844,4
84,1383365,2701.884766,5


In [5]:
ratings_df: pd.DataFrame = pd.read_excel(RATINGS_PATH)

s28_ratings: pd.DataFrame = ratings_df[
    ratings_df["Participant_id"] == 28
].copy()

s28_ratings = s28_ratings.sort_values("Trial").reset_index(drop=True)
s28_ratings["start_sec"] = s28_ratings["Start_time"] / 10000

event4_df: pd.DataFrame = df_events[df_events["code"] == 4].copy()
event4_df = event4_df.reset_index(drop=True)
event4_df["event4_index"] = event4_df.index + 1

print("ratings rows:", len(s28_ratings))
print("event 4 rows:", len(event4_df))

display(s28_ratings[["Trial", "Experiment_id", "Start_time", "start_sec"]].head(10))
display(event4_df.head(10))
display(s28_ratings[["Trial", "Experiment_id", "Start_time", "start_sec"]].tail(10))
display(event4_df.tail(10))

ratings rows: 40
event 4 rows: 37


,Trial,Experiment_id,Start_time,start_sec
0,1,12,1903620,190.3620
1,2,34,2764551,276.4551
2,3,2,3597195,359.7195
3,4,7,4400765,440.0765
4,5,21,5192388,519.2388
5,6,22,6023875,602.3875
6,7,32,6865767,686.5767
7,8,33,7706955,770.6955
8,9,15,8486596,848.6596
9,10,17,9307588,930.7588


,sample,time_sec,code,event4_index
0,112371,219.474609,4,1
1,156439,305.544922,4,2
2,199082,388.832031,4,3
3,240224,469.187500,4,4
4,280743,548.326172,4,5
5,323327,631.498047,4,6
6,366430,715.683594,4,7
7,409500,799.804688,4,8
8,449405,877.744141,4,9
9,491451,959.865234,4,10


,Trial,Experiment_id,Start_time,start_sec
30,31,20,27553255,2755.3255
31,32,13,28414898,2841.4898
32,33,31,29217685,2921.7685
33,34,39,29988757,2998.8757
34,35,40,30789484,3078.9484
35,36,5,31547488,3154.7488
36,37,29,32318627,3231.8627
37,38,19,33083309,3308.3309
38,39,28,33898368,3389.8368
39,40,36,34679087,3467.9087


,sample,time_sec,code,event4_index
27,1392416,2719.562500,4,28
28,1436543,2805.748047,4,29
29,1477634,2886.003906,4,30
30,1517112,2963.109375,4,31
31,1558109,3043.181641,4,32
32,1596931,3119.005859,4,33
33,1636401,3196.095703,4,34
34,1675564,3272.585938,4,35
35,1717294,3354.089844,4,36
36,1757268,3432.164062,4,37


In [6]:
comparison_rows: list[dict[str, float | int]] = []

for rating_index, rating_row in s28_ratings.iterrows():
    trial: int = int(rating_row["Trial"])
    rating_start_sec: float = float(rating_row["start_sec"])

    for event_index, event_row in event4_df.iterrows():
        event4_time_sec: float = float(event_row["time_sec"])
        diff_sec: float = event4_time_sec - rating_start_sec

        if 0.0 <= diff_sec <= 80.0:
            comparison_rows.append(
                {
                    "trial": trial,
                    "rating_index": rating_index + 1,
                    "event4_index": event_index + 1,
                    "rating_start_sec": rating_start_sec,
                    "event4_time_sec": event4_time_sec,
                    "diff_sec": diff_sec,
                }
            )

comparison_df: pd.DataFrame = pd.DataFrame(comparison_rows)

display(comparison_df.head(50))
display(comparison_df.tail(50))

,trial,rating_index,event4_index,rating_start_sec,event4_time_sec,diff_sec
0,1,1,1,190.3620,219.474609,29.112609
1,2,2,2,276.4551,305.544922,29.089822
2,3,3,3,359.7195,388.832031,29.112531
3,4,4,4,440.0765,469.187500,29.111000
4,5,5,5,519.2388,548.326172,29.087372
5,6,6,6,602.3875,631.498047,29.110547
6,7,7,7,686.5767,715.683594,29.106894
7,8,8,8,770.6955,799.804688,29.109187
8,9,9,9,848.6596,877.744141,29.084541
9,10,10,10,930.7588,959.865234,29.106434


,trial,rating_index,event4_index,rating_start_sec,event4_time_sec,diff_sec
0,1,1,1,190.3620,219.474609,29.112609
1,2,2,2,276.4551,305.544922,29.089822
2,3,3,3,359.7195,388.832031,29.112531
3,4,4,4,440.0765,469.187500,29.111000
4,5,5,5,519.2388,548.326172,29.087372
5,6,6,6,602.3875,631.498047,29.110547
6,7,7,7,686.5767,715.683594,29.106894
7,8,8,8,770.6955,799.804688,29.109187
8,9,9,9,848.6596,877.744141,29.084541
9,10,10,10,930.7588,959.865234,29.106434


In [7]:
direct_rows: list[dict[str, float | int | None]] = []

max_len: int = max(len(s28_ratings), len(event4_df))

for index in range(max_len):
    rating_start_sec: float | None = None
    event4_time_sec: float | None = None
    diff_sec: float | None = None

    if index < len(s28_ratings):
        rating_start_sec = float(s28_ratings.iloc[index]["start_sec"])

    if index < len(event4_df):
        event4_time_sec = float(event4_df.iloc[index]["time_sec"])

    if rating_start_sec is not None and event4_time_sec is not None:
        diff_sec = event4_time_sec - rating_start_sec

    direct_rows.append(
        {
            "index": index + 1,
            "trial": int(s28_ratings.iloc[index]["Trial"]) if index < len(s28_ratings) else None,
            "rating_start_sec": rating_start_sec,
            "event4_time_sec": event4_time_sec,
            "diff_sec": diff_sec,
        }
    )

direct_df: pd.DataFrame = pd.DataFrame(direct_rows)

display(direct_df)

,index,trial,rating_start_sec,event4_time_sec,diff_sec
0,1,1,190.3620,219.474609,29.112609
1,2,2,276.4551,305.544922,29.089822
2,3,3,359.7195,388.832031,29.112531
3,4,4,440.0765,469.187500,29.111000
4,5,5,519.2388,548.326172,29.087372
5,6,6,602.3875,631.498047,29.110547
6,7,7,686.5767,715.683594,29.106894
7,8,8,770.6955,799.804688,29.109187
8,9,9,848.6596,877.744141,29.084541
9,10,10,930.7588,959.865234,29.106434


In [8]:
# Zona donde la correspondencia se rompe: entre trial 23 y 27 aprox.
window_events: pd.DataFrame = df_events[
    (df_events["time_sec"] >= 2000)
    & (df_events["time_sec"] <= 2500)
].copy()

display(window_events)

,sample,time_sec,code
68,1038938,2029.175781,5
69,1048365,2047.587891,3
70,1050949,2052.634766,4
71,1081684,2112.664062,5
72,1091315,2131.474609,3
73,1227191,2396.857422,3
74,1229776,2401.906250,4
75,1260519,2461.951172,5
76,1266422,2473.480469,3
77,1269007,2478.529297,4


In [10]:
display(
    s28_ratings.loc[20:28, ["Trial", "Experiment_id", "Start_time", "start_sec"]]
)

,Trial,Experiment_id,Start_time,start_sec
20,21,37,18556262,1855.6262
21,22,16,19400371,1940.0371
22,23,4,20235357,2023.5357
23,24,26,21074241,2107.4241
24,25,25,21886296,2188.6296
25,26,14,22701010,2270.1010
26,27,23,24376444,2437.6444
27,28,8,25142677,2514.2677
28,29,10,25978062,2597.8062


In [11]:
# Offset estable observado en los primeros 23 trials.
stable_offset: float = float(direct_df.loc[0:22, "diff_sec"].median())

print("stable_offset:", stable_offset)

s28_check: pd.DataFrame = s28_ratings.copy()
s28_check["estimated_event4_sec"] = s28_check["start_sec"] + stable_offset
s28_check["estimated_event5_sec"] = s28_check["estimated_event4_sec"] + 60.0
s28_check["fits_in_bdf"] = s28_check["estimated_event5_sec"] <= (raw.n_times / raw.info["sfreq"])

display(
    s28_check[["Trial", "Experiment_id", "start_sec", "estimated_event4_sec", "estimated_event5_sec", "fits_in_bdf"]]
)

stable_offset: 29.100362500000074


,Trial,Experiment_id,start_sec,estimated_event4_sec,estimated_event5_sec,fits_in_bdf
0,1,12,190.3620,219.462363,279.462363,True
1,2,34,276.4551,305.555463,365.555463,True
2,3,2,359.7195,388.819863,448.819863,True
3,4,7,440.0765,469.176863,529.176863,True
4,5,21,519.2388,548.339163,608.339163,True
5,6,22,602.3875,631.487863,691.487863,True
6,7,32,686.5767,715.677063,775.677063,True
7,8,33,770.6955,799.795863,859.795863,True
8,9,15,848.6596,877.759963,937.759963,True
9,10,17,930.7588,959.859163,1019.859163,True
